# 使用 MobileNetV2 进行迁移学习（Transfer Learning with MobileNetV2）

欢迎学习本周的迁移学习作业！我们将使用一个预训练的卷积网络，建立“羊驼 / 非羊驼”二分类器。

![原作业：羊驼](images/alpaca.png)

预训练模型是在大型数据集上训练后保存的网络。复用它已经学到的特征，可以降低新任务的训练成本。MobileNetV2 面向计算资源有限的场景设计；本作业使用 ImageNet 预训练权重。原文将整个 ImageNet 数据库规模和 1000 类预训练任务合在一起介绍；这里明确采用 torchvision 的 ImageNet-1K 权重。

完成本作业后，你应能：

- 从图片目录创建数据集，并划分训练集与验证集。
- 进行图片预处理、随机水平翻转和旋转。
- 替换预训练模型的分类头，冻结主干后训练二分类器。
- 微调网络末尾的层，观察验证表现是否改善。

本文件按照同目录 `Transfer_learning_with_MobileNet_v1.ipynb` 的章节及三个练习顺序翻译改编，给出完整 PyTorch 参考实现。保留羊驼任务、160×160 输入、32 的 batch size，以及先训练 5 轮再微调 5 轮的设置。框架差异在对应位置说明；不要求复现 TensorFlow 自动评分器的逐项输出。

目录：1 导入工具与创建数据集 → 2 预处理与增强（练习 1）→ 3 MobileNetV2 迁移学习 → 3.1 块内结构与 1000 类预测 → 3.2 冻结训练（练习 2）→ 3.3 微调（练习 3）。

## 1 - 导入工具（Packages）

选择安装了 `torch、torchvision、numpy、Pillow、IPython` 的 Python 内核，从上到下执行。首次使用修改版请重启内核，让下方 MKL 兼容设置先于数值库导入生效。

全部图像展示、增强预览及曲线都使用 PIL 静态图片，不导入 Matplotlib，避开此前 `canvas.print_png()` 卡住的问题。图片在 PyTorch 中使用 NCHW，而原版 TensorFlow 使用 NHWC。

In [1]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")

from pathlib import Path
from io import BytesIO
import time
import random
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from PIL import Image, ImageDraw
from IPython.display import display, Image as PNGImage
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "设备:", device, flush=True)

def find_assignment_dir():
    name = "Transfer Learning with MobileNet"
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / name,
                          parent / "C4 - Convolutional Neural Networks/Week 2" / name):
            if (candidate / "Transfer_learning_with_MobileNet_v1.ipynb").is_file():
                return candidate
    raise FileNotFoundError("请在本项目内启动 Jupyter，或手动指定作业目录。")

ASSIGNMENT_DIR = find_assignment_dir()
BATCH_SIZE = 32
IMG_SIZE = (160, 160)
WEIGHTS = MobileNet_V2_Weights.IMAGENET1K_V1

def display_grid(images, captions=None, columns=3):
    # 用 PIL 拼接图片；不依赖 plt.subplot、plt.show 或交互式后端。
    images = list(images)
    if not images:
        return
    captions = captions if captions is not None else [""] * len(images)
    rows = (len(images) + columns - 1) // columns
    canvas = Image.new("RGB", (columns * 180, rows * 200), "white")
    draw = ImageDraw.Draw(canvas)
    for i, image in enumerate(images):
        x, y = (i % columns) * 180, (i // columns) * 200
        canvas.paste(image.convert("RGB").resize(IMG_SIZE), (x + 10, y))
        draw.text((x + 10, y + 165), str(captions[i]), fill="black")
    with BytesIO() as stream:
        canvas.save(stream, format="PNG")
        display(PNGImage(data=stream.getvalue()))
    canvas.close()

PyTorch: 2.7.1+cu118 设备: cuda


### 1.1 - 创建数据集，并划分为训练集与验证集

原版使用 `image_dataset_from_directory` 从磁盘读取图片，并设置 `validation_split=0.2`。训练和验证使用相同种子，以保证划分一致、不重叠。PyTorch 对应使用 `ImageFolder` 读取类别子目录，再用一份固定打乱的索引划分 80% / 20%。

当前项目尚未提供羊驼训练数据。请把原课程的 `dataset` 放到当前作业目录，或修改 `DATA_DIR`：

```text
Transfer Learning with MobileNet/
  dataset/
    alpaca/        羊驼图片
    not_alpaca/    非羊驼图片
```

`images/` 是讲解插图，不是训练数据。缺少数据时，下面会明确提示并跳过真实训练；增强示例、结构测试和绘图自检仍能运行。

类别编号按目录名排序分配，必须查看 `class_to_idx`。若目录名如上，通常 `alpaca=0、not_alpaca=1`；此时 Sigmoid 输出的是“非羊驼”的概率，不要把类别意义倒过来。沿用原版训练/验证划分，这里不另外声称拥有独立测试集。

In [2]:
DATA_DIR = ASSIGNMENT_DIR / "dataset"  # 可改成你的数据集绝对路径。
DATA_READY = DATA_DIR.is_dir()
raw_dataset = None
class_names = []
class_to_idx = {}
train_indices = val_indices = []

if DATA_READY:
    raw_dataset = datasets.ImageFolder(DATA_DIR)  # 暂不变换，返回 PIL 图片。
    class_names = raw_dataset.classes
    class_to_idx = raw_dataset.class_to_idx
    if len(class_names) != 2:
        raise ValueError(f"需要两个类别目录，实际为 {class_names}")
    indices = np.random.default_rng(SEED).permutation(len(raw_dataset))
    n_val = int(len(indices) * 0.2)
    if n_val < 2 or len(indices) - n_val < 2:
        raise ValueError("图片太少，无法合理划分两个类别的训练/验证集。")
    # 一份索引只划分一次，避免分别随机划分造成数据重叠。
    val_indices = indices[:n_val].tolist()
    train_indices = indices[n_val:].tolist()
    assert not set(train_indices) & set(val_indices)
    for name, selected in (("训练", train_indices), ("验证", val_indices)):
        if {raw_dataset.targets[i] for i in selected} != {0, 1}:
            raise ValueError(f"{name}划分缺少某个类别，请补充数据或调整划分种子。")
    print("类别映射:", class_to_idx)
    print("训练/验证图片数:", len(train_indices), len(val_indices))
else:
    print("缺少羊驼数据集:", DATA_DIR, flush=True)
    print("将跳过真实预测、训练和微调；请补齐数据后重新运行。", flush=True)

缺少羊驼数据集: D:\ACMpractice\pytorch\learn_pytorch_WuEnDa\C4 - Convolutional Neural Networks\Week 2\Transfer Learning with MobileNet\dataset
将跳过真实预测、训练和微调；请补齐数据后重新运行。


像原版一样，先观察九张训练图片及类别，确认读取结果正确。

In [ ]:
if DATA_READY:
    samples = [raw_dataset[i] for i in train_indices[:9]]
    display_grid([image for image, label in samples],
                 [class_names[label] for image, label in samples])
else:
    print("没有训练样本，跳过训练集九宫格。")

## 2 - 预处理与增强训练数据（Preprocess and Augment Training Data）

原文首先介绍 `prefetch()`：预先准备下一批数据，可以减少训练等待磁盘读取的时间；`AUTOTUNE` 自动调整预取配置。PyTorch 通常通过 DataLoader 工作进程等机制安排加载，但不是把这两个 TensorFlow API 原样替换。

这里为 Windows notebook 的稳定性设置 `num_workers=0`，不启用多进程预取。数据仍按需读取，不要求把所有原图一次装入内存。

为了增加训练样本的变化，我们对图片随机水平翻转和旋转。每次读同一张图片都可能得到不同版本，有助于降低对某种方向、姿态的依赖。验证集不随机增强，否则不同轮的评估难以比较。

### 练习 1 - data_augmenter

原版用 Sequential 串联 `RandomFlip('horizontal')` 和 `RandomRotation(0.2)`。这里用 `transforms.Compose` 串联相同类别的操作。

**旋转参数单位不同：** Keras 的 0.2 表示整圈的 20%，即随机角度约为 −72°～72°，所以 PyTorch 写 `RandomRotation(72)`，不能直接填 0.2。原版旋转默认反射填充；这里使用 PIL 双线性旋转和黑色填充，边界像素不会逐位一致，任务与增强顺序不变。

In [ ]:
def data_augmenter():
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(
            degrees=72,
            interpolation=transforms.InterpolationMode.BILINEAR,
            fill=0,
        ),
    ])

data_augmentation = data_augmenter()
assert isinstance(data_augmentation.transforms[0], transforms.RandomHorizontalFlip)
assert isinstance(data_augmentation.transforms[1], transforms.RandomRotation)
assert data_augmentation.transforms[1].degrees == [-72.0, 72.0]
print("练习 1 检查通过：随机水平翻转 + 随机旋转 ±72°。")

下面将同一张图随机增强九次，对应原版的九宫格。没有训练集时使用原作业羊驼插图，仅演示增强，不把插图当作训练集。

In [ ]:
if DATA_READY:
    first_image, _ = raw_dataset[train_indices[0]]
else:
    with Image.open(ASSIGNMENT_DIR / "images/alpaca.png") as source:
        first_image = source.convert("RGB")
    print("增强预览使用课程插图，不参与训练。")
first_image = first_image.resize(IMG_SIZE)
display_grid([data_augmentation(first_image) for _ in range(9)])

接着对输入归一化。原版 Keras 权重配合 `preprocess_input`，把像素转换到 [−1,1]。这里加载的是 **torchvision 权重**，要使用它对应的 RGB 均值和标准差：先除以 255，再按通道计算 `(x-mean)/std`。不能混用两个框架的预处理。

保留课程的 160×160 尺寸，不额外加入上一版的随机裁剪。torchvision 权重的标准评测变换使用 224×224 裁剪；MobileNetV2 的全局平均池化允许本作业采用 160×160，但这里不声称复现其 ImageNet 标准评测精度。

原版把增强放进模型；本版把增强放在训练数据变换中，计算顺序仍是“增强 → 标准化 → 主干”。训练、验证使用两个独立的 ImageFolder，保证验证集不被训练增强影响。保存模型参数并不会保存这些数据变换，预测时需要再次使用相同的确定性预处理。

In [ ]:
weight_preset = WEIGHTS.transforms()
MEAN, STD = weight_preset.mean, weight_preset.std
preprocess_input = transforms.Compose([
    transforms.ToTensor(),  # PIL RGB -> float32 (3,H,W)，范围 [0,1]。
    transforms.Normalize(mean=MEAN, std=STD),
])
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    data_augmentation,
    preprocess_input,
])
validation_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    preprocess_input,
])

train_loader = val_loader = None
if DATA_READY:
    train_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
    validation_dataset = datasets.ImageFolder(DATA_DIR, transform=validation_transform)
    assert train_dataset.samples == validation_dataset.samples == raw_dataset.samples
    train_loader = DataLoader(Subset(train_dataset, train_indices),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(Subset(validation_dataset, val_indices),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    image_batch, label_batch = next(iter(train_loader))
    print("图片/标签形状:", tuple(image_batch.shape), tuple(label_batch.shape))

## 3 - 使用 MobileNetV2 进行迁移学习

MobileNetV2 面向移动设备等低功耗场景，除了分类，也可作为检测和分割的特征提取主干。原版按 Keras 列出了 155 层；PyTorch 的模块组织方式不同，不直接比较 `len(layers)`。

原文强调三个特点：深度可分离卷积、较窄的输入输出瓶颈、瓶颈之间的快捷连接。

### 3.1 - MobileNetV2 卷积块内部（Inside a MobileNetV2 Convolutional Building Block）

普通卷积同时处理空间位置和通道混合，计算量较大。深度可分离卷积把任务分成两步：

1. **Depthwise convolution（逐通道卷积）**：每个通道独立做空间卷积。通道乘数为 1 时，每个输入通道对应一个卷积核；PyTorch 可用 `groups=in_channels` 表达。
2. **Pointwise convolution（逐点卷积）**：使用 1×1 卷积混合通道，生成所需数量的输出通道，而不只是固定合成一个通道。

![原作业：MobileNetV2 架构](images/mobilenetv2.png)

读图时关注块两端较窄、中间较宽的结构：通常先用 1×1 扩展通道，再用 3×3 depthwise 卷积处理空间信息，最后用 1×1 线性投影压缩通道。这叫倒残差（Inverted Residual），与之前 ResNet 的“先压缩再扩展”不同。扩展和 depthwise 后通常有 BN、ReLU6；最后投影后有 BN，但不接 ReLU，以免在低维表示上截断信息。

当步幅为 1 且输入输出通道相同时，快捷连接跨过主路径；下采样或通道不匹配时没有这种直接相加。

不计偏置时，普通卷积权重数为 $k^2C_{in}C_{out}$；depthwise + pointwise 为 $k^2C_{in}+C_{in}C_{out}$。完整倒残差块若包含扩展层，还要加上该层参数，不能只套后一个公式。

先像原版一样加载带 1000 类分类头的预训练模型，观察它的结构与预测。首次加载会下载 ImageNet 权重到 PyTorch 缓存，需要可用网络；下载失败会显示错误，不自动改用随机权重。

缺少羊驼训练数据时，这里仅用随机模型打印结构，不执行真实预测；随机模型只用于理解形状，不能代表迁移学习效果。

In [ ]:
base_model = mobilenet_v2(weights=WEIGHTS if DATA_READY else None).to(device)
base_model.eval()
if not DATA_READY:
    print("离线结构演示：当前模型使用随机权重，不用于判断准确率。")
print("特征模块数:", len(base_model.features))
print("分类头:", base_model.classifier)
print("参数数:", sum(p.numel() for p in base_model.parameters()))
print("末尾特征层:", base_model.features[-1])
# 完整结构较长，需要时取消下一行注释。
# print(base_model)

原版接着取一个 batch，查看 `(batch_size,1000)` 的输出，并解码每张图片概率最高的两个 ImageNet 类别。1000 类分类器的标签不是我们的“羊驼 / 非羊驼”，因此需要替换分类头。

PyTorch 模型输出 logits，先做 Softmax 再取前两项，类别名称来自权重的元数据。下方 `label_batch` 是数据集的真实 0/1 标签，不是概率。

In [ ]:
if DATA_READY:
    # 用确定性预处理展示，避免随机增强使演示图片难以辨认。
    image_batch = torch.stack([
        validation_transform(raw_dataset[i][0]) for i in train_indices[:min(9, BATCH_SIZE)]
    ]).to(device)
    label_batch = torch.tensor([raw_dataset.targets[i] for i in train_indices[:len(image_batch)]])
    with torch.no_grad():
        predictions = base_model(image_batch).softmax(dim=1)
    print("预测形状:", tuple(predictions.shape), "真实标签:", label_batch.tolist())
    probabilities, indices = predictions.topk(2, dim=1)
    categories = WEIGHTS.meta["categories"]
    for i in range(len(image_batch)):
        print(f"图片 {i}:", [(categories[k], round(p, 4))
              for k, p in zip(indices[i].tolist(), probabilities[i].tolist())])
else:
    with torch.no_grad():
        assert base_model(torch.zeros(2, 3, *IMG_SIZE, device=device)).shape == (2, 1000)
    print("1000 类输出形状检查通过；缺少数据，跳过预训练预测。")

### 3.2 - 冻结层并更换分类头（Layer Freezing with the Functional API）

![原作业：雪中的羊驼](images/snowalpaca.png)

原文将适配新任务分成三步：

1. 删除原来的分类头，对应 Keras 的 `include_top=False`。
2. 添加新分类头；二分类只需要一个输出神经元。
3. 冻结主干，只训练新分类头，并让主干中的 BN 使用已有统计量。

原版通过 Functional API 连接输入和输出。PyTorch 用 `nn.Module` 的 `forward` 描述同一计算链：预处理好的图片 → MobileNetV2 特征层 → 全局平均池化 → Dropout(0.2) → Linear(1280,1)。160×160 输入对应最后 `(N,1280,5,5)` 的特征图，平均池化后为 `(N,1280)`。

冻结需要区分：`requires_grad=False` 停止参数梯度更新；`eval()` 停止 BN 运行统计量更新。仅冻结参数不够。下面覆盖 `train()`，使每次开始训练时主干仍然保持评估模式，但分类头的 Dropout 正常处于训练模式。

### 练习 2 - alpaca_model

下面显式创建平均池化、Dropout 和一个输出神经元，对应原练习。数据增强已在前面的训练数据变换中执行，不在这里重复增强。

In [ ]:
class AlpacaModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.features = backbone.features  # 不取原来的 1000 类 classifier。
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=0.2)
        self.classifier = nn.Linear(backbone.last_channel, 1)
        for parameter in self.features.parameters():
            parameter.requires_grad = False
        self.features.eval()

    def train(self, mode=True):
        super().train(mode)
        # 与原版 base_model(x, training=False) 对应。
        # eval 不关闭 autograd：微调时解冻的卷积仍可计算梯度。
        self.features.eval()
        return self

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        return self.classifier(x)  # 一个 logit，不添加 Sigmoid。

def alpaca_model(backbone):
    return AlpacaModel(backbone)

model2 = alpaca_model(base_model).to(device)
# features 复用了同一主干；后面统一通过 model2.features 访问它。
del base_model
model2.eval()
with torch.no_grad():
    features = model2.features(torch.zeros(2, 3, *IMG_SIZE, device=device))
    assert features.shape == (2, 1280, 5, 5)
    assert model2(torch.zeros(2, 3, *IMG_SIZE, device=device)).shape == (2, 1)
assert sum(p.numel() for p in model2.parameters() if p.requires_grad) == 1281
print("练习 2 检查通过：仅分类头的 1280 个权重和 1 个偏置可训练。")

原版使用 Adam、学习率 0.01、带 logits 的二元交叉熵，并训练 5 轮。这里保留这些设置。如果你之后观察到训练明显震荡，可再尝试更小的学习率，但不要把原版输出当作固定标准答案。

对应关系：`BinaryCrossentropy(from_logits=True)` → `BCEWithLogitsLoss()`；标签转换为浮点 0/1，预测和标签都整理成 `(N,)`。准确率使用 `logit >= 0`，等价于 `sigmoid(logit) >= 0.5`，不能直接用 0.5 作为 logit 的阈值。

PyTorch 没有这里所需的 Keras `compile/fit` 训练封装，所以下面显式实现前向计算、损失、反向传播、更新参数与每轮验证，并打印 batch 进度。

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, label="验证"):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, count = 0.0, 0, 0
    started = time.perf_counter()
    print(f"[{label}] 开始，共 {len(loader)} 个 batch", flush=True)
    with torch.set_grad_enabled(training):
        for step, (xb, yb) in enumerate(loader, 1):
            xb = xb.to(device)
            yb = yb.to(device).float().reshape(-1)
            if not torch.all((yb == 0) | (yb == 1)).item():
                raise ValueError("BCE 标签必须为 0/1，请检查数据集类别。")
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(xb).reshape(-1)
            loss = criterion(logits, yb)
            if not torch.isfinite(loss).item():
                raise RuntimeError("损失出现 NaN/Inf，请检查输入和学习率。")
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += ((logits >= 0) == yb.bool()).sum().item()
            count += len(yb)
            if step == 1 or step % 10 == 0 or step == len(loader):
                print(f"[{label}] batch {step}/{len(loader)}, "
                      f"loss={total_loss/count:.4f}, 耗时 {time.perf_counter()-started:.1f}s", flush=True)
    if count == 0:
        raise ValueError("数据加载器为空。")
    return total_loss / count, correct / count

def fit(model, train_loader, val_loader, criterion, optimizer, epochs, start_epoch=0):
    records = {key: [] for key in ("loss", "accuracy", "val_loss", "val_accuracy")}
    for epoch in range(start_epoch + 1, start_epoch + epochs + 1):
        print(f"\nEpoch {epoch}/{start_epoch + epochs}", flush=True)
        loss, acc = run_epoch(model, train_loader, criterion, optimizer, label="训练")
        vl, va = run_epoch(model, val_loader, criterion, label="验证")
        for key, value in zip(records, (loss, acc, vl, va)):
            records[key].append(value)
        print(f"本轮完成: loss={loss:.4f}, accuracy={acc:.3f}, "
              f"val_loss={vl:.4f}, val_accuracy={va:.3f}", flush=True)
    return records

base_learning_rate = 0.01
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    [p for p in model2.parameters() if p.requires_grad], lr=base_learning_rate)
initial_epochs = 5
history = None
if DATA_READY:
    history = fit(model2, train_loader, val_loader, criterion, optimizer, epochs=initial_epochs)
else:
    print("缺少羊驼数据：跳过真实分类头训练。")

像原版一样，先绘制冻结训练阶段的准确率和损失。下面统一使用 PIL，自动按数据范围设置纵轴，不截掉超过 1 的损失，也不在准确率记录前人为添加一个零。

绘图函数能显示微调开始位置。缺少真实训练记录时，使用明确标记的演示记录自检图片能否生成；演示数值不代表模型表现。

In [ ]:
def plot_history(history, fine_tune_start=None):
    """使用 PIL 绘制静态曲线，绕过本机 Matplotlib PNG 渲染卡顿。"""
    from PIL import ImageDraw
    import math
    import time

    started = time.perf_counter()
    for key, ylabel in (("loss", "Loss"), ("accuracy", "Accuracy")):
        print(f"[绘图] 开始绘制 {ylabel}……", flush=True)
        train = [float(v) for v in history[key]]
        validation = [float(v) for v in history["val_" + key]]
        if not train or not validation:
            print(f"[绘图] {ylabel} 无数据，跳过。", flush=True)
            continue
        values = train + validation
        if not all(math.isfinite(v) for v in values):
            raise ValueError(f"{ylabel} 中存在 NaN 或无穷大，请检查训练记录。")

        # PIL 直接绘制像素，不使用 Matplotlib、Agg 或 pyplot。
        picture = Image.new("RGB", (800, 450), "white")
        draw = ImageDraw.Draw(picture)
        left, top, right, bottom = 85, 55, 745, 380
        low, high = min(values), max(values)
        margin = max((high - low) * 0.1, 0.01)
        low, high = low - margin, high + margin
        count = max(len(train), len(validation))

        draw.text((left, 15), "Model " + ylabel, fill="black")
        draw.text((450, 15), "Train", fill="blue")
        draw.text((550, 15), "Validation", fill="red")
        draw.text((370, 420), "Epoch", fill="black")

        # 纵轴刻度与网格。
        for i in range(6):
            value = low + (high - low) * i / 5
            y = bottom - (bottom - top) * i / 5
            draw.line([(left, y), (right, y)], fill="#dddddd")
            draw.text((5, y - 5), f"{value:.3f}", fill="black")
        draw.line([(left, top), (left, bottom), (right, bottom)],
                  fill="black", width=2)

        # 最多显示 6 个 epoch 刻度；也支持只有一轮的记录。
        for index in sorted({round(i * (count - 1) / 5) for i in range(6)}):
            x = left + index / max(count - 1, 1) * (right - left)
            draw.text((x - 5, bottom + 10), str(index + 1), fill="black")

        # fine_tune_start 是第几轮开始微调；边界画在上一轮与本轮之间。
        if fine_tune_start is not None and 1 < fine_tune_start <= count:
            boundary = left + (fine_tune_start - 1.5) / max(count - 1, 1) * (right - left)
            draw.line([(boundary, top), (boundary, bottom)], fill="green", width=2)
            draw.text((left, 35), f"Fine-tuning starts at epoch {fine_tune_start}", fill="green")

        for series, color in ((train, "blue"), (validation, "red")):
            points = [
                (left + i / max(count - 1, 1) * (right - left),
                 bottom - (value - low) / (high - low) * (bottom - top))
                for i, value in enumerate(series)
            ]
            if len(points) > 1:
                draw.line(points, fill=color, width=3)
            for x, y in points:
                draw.ellipse((x - 3, y - 3, x + 3, y + 3), fill=color)

        print(f"[绘图] {ylabel} 绘制完成，开始显示……", flush=True)
        # 显式编码 PNG，避免依赖交互式绘图后端。
        with BytesIO() as stream:
            picture.save(stream, format="PNG")
            display(PNGImage(data=stream.getvalue()))
        picture.close()
        print(f"[绘图] {ylabel} 显示调用完成。", flush=True)
    print(f"[绘图] 全部完成，用时 {time.perf_counter() - started:.2f} 秒。", flush=True)

In [ ]:
if history is not None:
    plot_history(history)
else:
    print("以下仅为绘图自检示例，不是训练结果。", flush=True)
    demo_history = {"loss": [0.8, 0.6, 0.5], "accuracy": [0.5, 0.65, 0.7],
                    "val_loss": [0.85, 0.7, 0.65], "val_accuracy": [0.5, 0.6, 0.65]}
    plot_history(demo_history)

### 3.3 - 微调模型（Fine-tuning the Model）

原文接下来尝试微调末尾层：先解冻一部分深层参数，再以更小的学习率继续训练。浅层常提取边缘等通用特征，深层更贴近具体任务，例如毛发、耳朵等特征。让深层适应新数据，可能改善识别效果。

解冻位置不是唯一答案。原版选择 Keras 层编号 126；PyTorch 将多层组织成 `features` 模块，不能把数字 126 直接照抄。下面打印模块后，从 `features[14:]` 开始微调，这是按尾部模块选取的学习设置，不声称与 Keras 第 126 层逐层等价。

### 练习 3 - 解冻末尾层，降低学习率

按照原文：较早层继续冻结、末尾层允许更新、学习率变为原来的 0.1 倍、仍使用带 logits 的二元交叉熵。重新创建优化器，让新解冻参数进入优化器。

本版和原版的 `base_model(..., training=False)` 一样，保持 BN 运行均值与方差不变；解冻区域内 BN 的 gamma/beta 仍可学习。`eval()` 与“参数完全不能训练”不是同一件事。

In [ ]:
fine_tune_at = 14
for i, layer in enumerate(model2.features):
    print(i, type(layer).__name__, "将解冻" if i >= fine_tune_at else "继续冻结")

def unfreeze_tail(model, start=14):
    if not 0 <= start < len(model.features):
        raise ValueError("解冻起点超出特征模块范围。")
    for i, layer in enumerate(model.features):
        for parameter in layer.parameters():
            parameter.requires_grad = i >= start
    model.features.eval()  # 保持 BN 统计量；不阻止解冻参数计算梯度。

history_fine = None
if history is not None:
    unfreeze_tail(model2, fine_tune_at)
    optimizer = torch.optim.Adam(
        [p for p in model2.parameters() if p.requires_grad], lr=base_learning_rate * 0.1)
    assert optimizer.param_groups[0]["lr"] == base_learning_rate / 10
    assert all(not p.requires_grad for p in model2.features[0].parameters())
    assert any(p.requires_grad for p in model2.features[-1].parameters())
    print("练习 3 检查通过，可训练参数:",
          sum(p.numel() for p in model2.parameters() if p.requires_grad))
else:
    print("尚未完成真实分类头训练，跳过微调配置。")

原版意图是再训练 5 轮。这里明确从第 6 轮开始，到第 10 轮结束，避免原代码使用最后一轮零基索引而重复一轮的歧义。

微调不保证一定提升准确率；应比较训练/验证曲线，观察是否过拟合或学习率过大，而不是预先宣称效果改善。

In [ ]:
fine_tune_epochs = 5
if history is not None:
    history_fine = fit(model2, train_loader, val_loader, criterion, optimizer,
                       epochs=fine_tune_epochs, start_epoch=initial_epochs)
else:
    print("缺少训练结果，跳过真实微调。")

In [ ]:
if history is not None and history_fine is not None:
    total_history = {key: history[key] + history_fine[key] for key in history}
    plot_history(total_history, fine_tune_start=initial_epochs + 1)
    val_loss, val_acc = run_epoch(model2, val_loader, criterion)
    print(f"最终验证集：loss={val_loss:.4f}, accuracy={val_acc:.3f}")
else:
    print("尚无真实的两阶段记录，跳过最终训练曲线。")

### 原作业小结

- 适配新任务：移除原分类头，添加新分类头，先只训练新层。
- 冻结主干时同时处理 BN 运行统计量，避免意外改变预训练模型的状态。
- 再以较小学习率微调末尾层，使高层特征适应新任务，并通过验证集检查效果。

到这里，你已按原作业顺序完成数据集创建、数据增强、冻结训练和微调。缺少数据时只能完成结构和绘图学习，不能算完成真实迁移学习训练。

### 补充：用自己的图片预测（可选）

下面是本地学习补充，不改变前面的课程步骤。个人图片使用与验证集一致的预处理。Sigmoid 是类别 1 的概率，类别名称必须通过映射读取；若 `alpaca=0`，羊驼概率为 `1-p1`。

In [ ]:
MY_IMAGE = None  # 例如 r"C:\Users\21467\Desktop\alpaca.jpg"
if history is not None and MY_IMAGE is not None:
    with Image.open(MY_IMAGE) as source:
        image = source.convert("RGB")
    x = validation_transform(image).unsqueeze(0).to(device)
    model2.eval()
    with torch.no_grad():
        p1 = model2(x).sigmoid().item()
    display_grid([image])
    print({class_names[0]: 1 - p1, class_names[1]: p1})
else:
    print("个人图片预测已跳过：需先完成真实训练并设置图片路径。")

SAVE_MODEL = False
if SAVE_MODEL and history is not None:
    destination = ASSIGNMENT_DIR / "alpaca_mobilenetv2_zh.pt"
    if destination.exists():
        raise FileExistsError("请换一个文件名，避免覆盖已有权重。")
    torch.save({"state_dict": model2.state_dict(), "class_to_idx": class_to_idx,
                "image_size": IMG_SIZE, "mean": MEAN, "std": STD,
                "pretrained_weights": "IMAGENET1K_V1"}, destination)
    print("已保存:", destination)

直接改编来源：同目录 `Transfer_learning_with_MobileNet_v1.ipynb`，插图来自其 `images` 文件夹。主要框架调整为 NCHW、torchvision 权重标准化、Compose/DataLoader、显式训练循环、尾部模块索引和 PIL 绘图。

参考：[torchvision MobileNetV2](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v2.html)。本文件供本地学习，不用于原 TensorFlow 自动评分器。